# Kaze で Qwen3.8-27B を動かす

Triton 製の推論エンジン [Kaze](https://github.com/sahenjp/kaze) で、Qwen3.8-27B（hybrid 注意）を
int4 重みで読み込んで生成します。**L4(24GB) か A100(40GB) を選んでください。**


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q torch triton transformers safetensors huggingface_hub
!pip install -q git+https://github.com/sahenjp/kaze

## 1. 先に「軽い経路」が動くか確かめる（数十秒・GPU 1GB 未満）

27B の重みは 55GB あるので、まず同じ実装を小さな乱数モデルで通します。
ここが通れば、あとは重みのサイズの問題だけです。


In [ ]:
import torch
from kaze.qwen38 import Qwen38Config, Qwen38Model, generate

cfg = Qwen38Config({"vocab_size": 256, "hidden_size": 128, "intermediate_size": 256,
                    "num_hidden_layers": 4, "num_attention_heads": 4, "num_key_value_heads": 2,
                    "head_dim": 32, "rms_norm_eps": 1e-6,
                    "rope_parameters": {"rope_theta": 10000.0},
                    "partial_rotary_factor": 0.5, "attn_output_gate": True,
                    "output_gate_type": "swish", "linear_conv_kernel_dim": 4,
                    "linear_key_head_dim": 16, "linear_value_head_dim": 16,
                    "linear_num_key_heads": 2, "linear_num_value_heads": 4,
                    "layer_types": ["linear_attention", "full_attention"] * 2})
torch.manual_seed(0)
sd = {"model.embed_tokens.weight": torch.randn(256, 128).half(),
      "model.norm.weight": torch.ones(128).half(),
      "lm_head.weight": torch.randn(256, 128).half()}
for i, lt in enumerate(cfg.layer_types):
    p = f"model.layers.{i}"
    sd[p + ".input_layernorm.weight"] = torch.ones(128).half()
    sd[p + ".post_attention_layernorm.weight"] = torch.ones(128).half()
    sd[p + ".mlp.gate_proj.weight"] = torch.randn(256, 128).half() * 0.02
    sd[p + ".mlp.up_proj.weight"] = torch.randn(256, 128).half() * 0.02
    sd[p + ".mlp.down_proj.weight"] = torch.randn(128, 256).half() * 0.02
    if lt == "full_attention":
        sd[p + ".self_attn.q_proj.weight"] = torch.randn(256, 128).half() * 0.02
        sd[p + ".self_attn.k_proj.weight"] = torch.randn(64, 128).half() * 0.02
        sd[p + ".self_attn.v_proj.weight"] = torch.randn(64, 128).half() * 0.02
        sd[p + ".self_attn.o_proj.weight"] = torch.randn(128, 128).half() * 0.02
        sd[p + ".self_attn.q_norm.weight"] = torch.ones(32).half()
        sd[p + ".self_attn.k_norm.weight"] = torch.ones(32).half()
    else:
        sd[p + ".linear_attn.in_proj_qkv.weight"] = torch.randn(128, 128).half() * 0.02
        sd[p + ".linear_attn.in_proj_z.weight"] = torch.randn(64, 128).half() * 0.02
        sd[p + ".linear_attn.in_proj_b.weight"] = torch.randn(4, 128).half() * 0.02
        sd[p + ".linear_attn.in_proj_a.weight"] = torch.randn(4, 128).half() * 0.02
        sd[p + ".linear_attn.conv1d.weight"] = torch.randn(128, 1, 4).half() * 0.02
        sd[p + ".linear_attn.A_log"] = torch.zeros(4)
        sd[p + ".linear_attn.dt_bias"] = torch.zeros(4)
        sd[p + ".linear_attn.norm.weight"] = torch.ones(16).half()
        sd[p + ".linear_attn.out_proj.weight"] = torch.randn(128, 64).half() * 0.02

small = Qwen38Model(cfg, sd, dtype=torch.float16, device="cuda", kind="int4")
print("small ok ->", generate(small, [5, 12, 33], max_tokens=4))

## 2. 27B を落とす（55GB・10分ほど）

ディスクの空きが 60GB 以上あることを確認してから実行してください。


In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download('Qwen/Qwen3.8-27B')
path

In [ ]:
from transformers import AutoTokenizer
from kaze.qwen38 import load_qwen38

tok = AutoTokenizer.from_pretrained(path)
model = load_qwen38(path, kind='int4', dtype=torch.float16)   # int8 なら約27GB
print('GPU', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

In [ ]:
import time

prompt = '深海の調査船について、200字ほどで説明してください。'
ids = tok(prompt).input_ids
t0 = time.perf_counter()
out = generate(model, ids, max_tokens=64, temperature=0.0,
               eos_token_ids=(tok.eos_token_id,))
dt = time.perf_counter() - t0
print(tok.decode(out, skip_special_tokens=True))
print(f'--- {len(out)} tokens in {dt:.1f}s ({len(out)/dt:.1f} tok/s)')

## メモ

- decode は 1 トークンずつ Python から回しているため遅いです（CUDA Graph 未対応）。
  27B では 1 トークン 200ms 前後（5 tok/s 程度）になる見込みです。
- 量子化は読み込み時にその場で行います（変換済みファイルは不要）。
- メモリが足りないときは `kind='int4'` のまま `max_tokens` を小さくしてください。
  int8 にすると 27GB 必要になります。
